# Evaluacion Parcial 3 - Programacion para la ciencia de datos

- CARLOS ACEVEDO ALBERTO CANGAS (EL CAPITAN)

- Jean Bizama

- Joaquìn Muñoz

# 1. Importacion de librerias

In [73]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder, StandardScaler
from sklearn.model_selection import cross_val_score, StratifiedKFold, GridSearchCV, RandomizedSearchCV, train_test_split
from sklearn.svm import SVC
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import davies_bouldin_score, silhouette_score, accuracy_score, f1_score, precision_score, recall_score, classification_report, confusion_matrix
from sklearn.utils import resample
import warnings
from sklearn.exceptions import UndefinedMetricWarning
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=UndefinedMetricWarning)

# 2. Generacion de DataFrame

In [74]:
df = pd.read_csv('/content/googleplaystore.csv')
df = df.drop_duplicates().copy() #Se eliminan los datos duplicados en caso de haber
df.head(3)

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19M,"10,000+",Free,0,Everyone,Art & Design,"January 7, 2018",1.0.0,4.0.3 and up
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,"500,000+",Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,8.7M,"5,000,000+",Free,0,Everyone,Art & Design,"August 1, 2018",1.2.4,4.0.3 and up


In [75]:
#Dejamos los datos mas relevantes para predecir el Rating de una app
df_relevante = df[["Rating","Category", "Reviews", "Size", "Type", "Price"]].copy()
df_relevante.head(3)

,Rating,Category,Reviews,Size,Type,Price
0,4.1,ART_AND_DESIGN,159,19M,Free,0
1,3.9,ART_AND_DESIGN,967,14M,Free,0
2,4.7,ART_AND_DESIGN,87510,8.7M,Free,0


In [76]:
#Durante la creacion del codigo descubrimos una fila con datos erroneos
print(df_relevante[df_relevante['Price'] == 'Everyone'])
#Conservamos el dataset sin esa fila
df_relevante = df_relevante[df_relevante['Price'] != 'Everyone']

       Rating Category Reviews    Size Type     Price
10472    19.0      1.9    3.0M  1,000+    0  Everyone


# 3. Limpieza del dataset relevante

In [77]:
#Es necesario hacer un tratamiento de datos nulos en las columnas 'Rating' y 'Type'
df_relevante.info()

<class 'pandas.core.frame.DataFrame'>
Index: 10357 entries, 0 to 10840
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Rating    8892 non-null   float64
 1   Category  10357 non-null  object 
 2   Reviews   10357 non-null  object 
 3   Size      10357 non-null  object 
 4   Type      10356 non-null  object 
 5   Price     10357 non-null  object 
dtypes: float64(1), object(5)
memory usage: 566.4+ KB


In [79]:
#Eliminamos la fila que tiene el nulo ya que solo es 1 dato
df_relevante.dropna(subset=['Type'], inplace=True)

mediana_rating = df['Rating'].median()
df_relevante['Rating'] = df['Rating'].fillna(mediana_rating)

#Limpiamos 'Price' Quitamos el símbolo '$' y convertimos a decimal
df_relevante['Price'] = df_relevante['Price'].astype(str).str.replace('$', '', regex=False).astype(float)
#Decidimos poner un umbral de precio en 50$, detectando como outliers a los de mayor precio
outliers_price = df_relevante[df_relevante['Price'] > 50]
df_relevante = df_relevante.drop(outliers_price.index)

#Limpiamos 'Reviews' Convirtiendolo directamente a entero
df_relevante['Reviews'] = df_relevante['Reviews'].astype(int)

#SIZE
#Estandarizar el size
for i, valor in enumerate(df_relevante['Size']):
    if pd.isnull(valor) or valor == 'Varies with device':
      df_relevante.iloc[i, df_relevante.columns.get_loc('Size')] = np.nan
    elif 'M' in str(valor):
      #Si son Mb, multiplicamos por 104.8576 (valor de 1 mb)
      df_relevante.iloc[i, df_relevante.columns.get_loc('Size')] = float(str(valor).replace('M', '')) * 1048576
    elif 'k' in str(valor):
      #Si son kb, multiplicamos por 1.024 (valor de 1kb)
      df_relevante.iloc[i, df_relevante.columns.get_loc('Size')] = float(str(valor).replace('k', '')) * 1024

#Convertimos explícitamente la columna 'Size' a numérica
df_relevante['Size'] = pd.to_numeric(df_relevante['Size'])

#Calculamos la mediana y rellenamos los nulos
mediana_size = df_relevante['Size'].median()
df_relevante['Size'] = df_relevante['Size'].fillna(mediana_size)

# 4. Encoding y estandarizacion de variables categoricas

In [82]:
#Encoding
le_category = LabelEncoder()
le_type = LabelEncoder()

# Transformamos el texto a identificadores numéricos
df_relevante['Category'] = le_category.fit_transform(df_relevante['Category'])
df_relevante['Type'] = le_type.fit_transform(df_relevante['Type'])

In [83]:
#Estandarizacion
#Seleccionamos las columnas que vamos a usar para predecir (todas menos Rating)
columnas_predictoras = ['Category', 'Reviews', 'Size', 'Type', 'Price']

scaler = StandardScaler()

#Estandarizamos los datos de las columnas predictoras
df_relevante[columnas_predictoras] = scaler.fit_transform(df_relevante[columnas_predictoras])

,Rating,Category,Reviews,Size,Type,Price
0,4.1,-2.002655,-0.150641,-0.051663,-0.278144,-0.172158
1,3.9,-2.002655,-0.150342,-0.289411,-0.278144,-0.172158
2,4.7,-2.002655,-0.118286,-0.541425,-0.278144,-0.172158
